# DepthwiseCNN — FFT-75 Benchmark (Kaggle)

End-to-end notebook for training and evaluating the **DepthwiseCNN** model family
on the FFT-75 file-fragment classification benchmark.

Reference paper: *File Fragment Type Classification Using Light-Weight Convolutional Neural Networks*

---

## Kaggle inputs required

| Path | Fragment size |
|------|---------------|
| `/kaggle/input/datasets/thegifman/fft-75-512-1/` | 512 B |
| `/kaggle/input/datasets/thegifman/fft-75-4096-1/` | 4096 B |

Each folder must contain `train.npz`, `val.npz`, `test.npz`.

---

## Notebook structure

| § | Description |
|---|-------------|
| 0 | Runtime config (the **only** cell you need to edit) |
| 1 | Environment setup + repo clone + package install |
| 2 | Dataset preparation (both 512 and 4096) |
| 3 | Dataset validation |
| 4 | Sanity pass (2 epochs, tiny subset) |
| 5 | Full training |
| 6 | Evaluation (test split) |
| 7 | Results summary + bundle |

---
## § 0 — Runtime configuration (edit here only)

In [ ]:
# =============================================================================
# § 0  RUNTIME CONFIGURATION
# =============================================================================

# Fragment size in bytes — 512 or 4096
FRAGMENT_SIZE = 512    # <-- change to 4096 for the 4096-byte scenario

# Architecture variant — 'dsc' | 'dsc_se' | 'm_dsc'
VARIANT = 'dsc'        # <-- change to select a different model variant

# Training hyperparameters
EPOCHS      = 50
BATCH_SIZE  = 256
LR          = 1e-3
SEED        = 42

# Sanity-pass settings (§ 4)
SANITY_EPOCHS  = 2
SANITY_SAMPLES = 2000

# GitHub repo
GITHUB_REPO  = 'https://github.com/yuvnahr/deepcarv.git'
BRANCH_NAME  = 'eval-depthwiseCNN'
GITHUB_TOKEN = ''  # optional PAT for private repos

print(f'Config: fragment_size={FRAGMENT_SIZE}, variant={VARIANT}')

---
## § 1 — Environment setup + repo clone + package install

Clones the repo and runs `pip install -e .` so that `src` and `benchmarks`
are importable as proper Python packages — no `sys.path` hacks needed.

In [ ]:

import gc
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORKING_DIR    = Path('/kaggle/working')
REPO_DIR       = WORKING_DIR / 'deepcarv'
CANONICAL_ROOT = WORKING_DIR / 'data' / 'FFT-75'
OUTPUTS_DIR    = WORKING_DIR / 'outputs'
CKPT_DIR       = WORKING_DIR / 'checkpoints'
LOGS_DIR       = WORKING_DIR / 'logs'

for d in (CANONICAL_ROOT, OUTPUTS_DIR, CKPT_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Clone repo (always fresh to ensure latest commits are pulled)
# ---------------------------------------------------------------------------
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = (
    f'https://{GITHUB_TOKEN}@github.com/yuvnahr/deepcarv.git'
    if GITHUB_TOKEN else GITHUB_REPO
)
print('Cloning repo ...')
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH_NAME,
     clone_url, str(REPO_DIR)],
    capture_output=True, text=True,
)
if result.returncode != 0:
    raise RuntimeError(f'git clone failed:\n{result.stderr}')
print('Clone complete.')



# ---------------------------------------------------------------------------
# Install remaining Python deps
# ---------------------------------------------------------------------------
def _pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

_pip('pyyaml>=6.0', 'scikit-learn>=1.5', 'pandas>=2.2',
     'matplotlib>=3.9', 'seaborn>=0.13', 'tqdm>=4.66', 'tabulate>=0.9')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')



# Tell DeepCarv path resolver where things live
os.environ['KAGGLE_RUNTIME']           = '1'
os.environ['DEEPCARV_DATA_ROOT']       = str(WORKING_DIR / 'data')
os.environ['DEEPCARV_OUTPUTS_DIR']     = str(OUTPUTS_DIR)
os.environ['DEEPCARV_CHECKPOINTS_DIR'] = str(CKPT_DIR)
os.environ['DEEPCARV_LOGS_DIR']        = str(LOGS_DIR)

print('\n§ 1 complete. ✓')

# Ensure Python sees the newly cloned files
import sys
import importlib
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()

# Verify imports work
import src.data.dataset  # noqa: F401
import benchmarks.DepthwiseCNN.scripts.train  # noqa: F401
print('Import check: src  ✓')
print('Import check: benchmarks  ✓')

gc.collect()

---
## § 2 — Dataset preparation

In [ ]:
import gc
import numpy as np
from pathlib import Path

WORKING_DIR    = Path('/kaggle/working')
CANONICAL_ROOT = WORKING_DIR / 'data' / 'FFT-75'
SPLITS = ('train', 'val', 'test')

# Exact Kaggle input paths
KAGGLE_ROOTS = {
    512:  Path('/kaggle/input/datasets/thegifman/fft-75-512-1'),
    4096: Path('/kaggle/input/datasets/thegifman/fft-75-4096-1'),
}

print('=== Kaggle input paths ===')
for fsize, root in KAGGLE_ROOTS.items():
    exists = root.exists()
    files  = [f.name for f in root.iterdir()] if exists else []
    print(f'  {fsize}B  exists={exists}  files={files}')


def _load_npz_arrays(path: Path):
    with np.load(str(path), allow_pickle=False) as data:
        keys = {k.lower(): k for k in data.files}
        if 'x' not in keys or 'y' not in keys:
            raise ValueError(f'{path.name}: expected x/y, found {list(data.files)}')
        return np.asarray(data[keys['x']]), np.asarray(data[keys['y']])


def _write_normalized_split(src: Path, dst: Path, fragment_size: int) -> None:
    X, y = _load_npz_arrays(src)
    if X.ndim == 1 and X.size % fragment_size == 0:
        X = X.reshape(-1, fragment_size)
    if X.ndim != 2 or X.shape[1] != fragment_size:
        raise ValueError(f'{src.name}: expected [N,{fragment_size}], got {X.shape}')
    if y.ndim != 1:
        y = y.reshape(-1)
    if len(X) != len(y):
        raise ValueError(f'{src.name}: len(X)={len(X)} != len(y)={len(y)}')
    np.savez_compressed(str(dst), x=X, y=y)


def prepare_dataset(fragment_size: int) -> Path:
    target_dir  = CANONICAL_ROOT / str(fragment_size)
    kaggle_root = KAGGLE_ROOTS[fragment_size]

    if all((target_dir / f'{s}.npz').exists() for s in SPLITS):
        print(f'  {fragment_size}B: already prepared')
        return target_dir

    if not kaggle_root.exists():
        raise FileNotFoundError(f'Kaggle input not found: {kaggle_root}')

    print(f'\nPreparing {fragment_size}B ...')
    target_dir.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        src = kaggle_root / f'{split}.npz'
        if not src.exists():
            raise FileNotFoundError(f'Missing: {src}')
        dst = target_dir / f'{split}.npz'
        _write_normalized_split(src, dst, fragment_size)
        X_tmp, y_tmp = _load_npz_arrays(dst)
        print(f'  {split}: {X_tmp.shape}  classes={len(set(y_tmp.tolist()))}')

    return target_dir


print('\n=== Preparing datasets ===')
for fsize in (512, 4096):
    prepare_dataset(fsize)

print('\n=== Canonical layout ===')
for fsize in (512, 4096):
    for split in SPLITS:
        p = CANONICAL_ROOT / str(fsize) / f'{split}.npz'
        print(f'  {fsize}/{split}.npz -> {p.exists()}')

gc.collect()
print('\n§ 2 complete. ✓')

---
## § 3 — Dataset validation

In [ ]:
import numpy as np
from pathlib import Path

CANONICAL_ROOT = Path('/kaggle/working/data/FFT-75')


def validate_npz(path: Path, frag_size: int) -> None:
    assert path.exists(), f'Missing: {path}'
    with np.load(str(path), allow_pickle=False) as data:
        assert set(data.files) == {'x', 'y'}, (
            f'{path.name}: expected {{x, y}}, got {set(data.files)}'
        )
        X, y = data['x'], data['y']
    assert X.ndim == 2 and X.shape[1] == frag_size
    assert y.ndim == 1 and len(y) == len(X)
    print(f'  OK  {path.name:<12}  N={len(X):>8,}  classes={int(y.max())+1}')


print(f'Validating {FRAGMENT_SIZE}B splits ...')
for split in ('train', 'val', 'test'):
    validate_npz(CANONICAL_ROOT / str(FRAGMENT_SIZE) / f'{split}.npz', FRAGMENT_SIZE)

print('\nDataset validation passed. ✓')

---
## § 4 — Sanity pass (2 epochs, tiny subset)

In [ ]:
import sys
if '/kaggle/working/deepcarv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deepcarv')

import gc
import logging
from pathlib import Path

import torch

from src.data.dataset import FragmentDataset, build_dataloader
from src.models.registry import build_model
from src.training.trainer import Trainer, TrainerConfig
from src.utils.seed import set_seed

CANONICAL_ROOT = Path('/kaggle/working/data/FFT-75')
OUTPUTS_DIR    = Path('/kaggle/working/outputs')

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(levelname)-8s %(message)s', force=True)

set_seed(SEED)

train_ds = FragmentDataset(
    root_dir=CANONICAL_ROOT, split='train',
    fragment_size=FRAGMENT_SIZE, cache=True, tiny_subset=SANITY_SAMPLES,
)
val_ds = FragmentDataset(
    root_dir=CANONICAL_ROOT, split='val',
    fragment_size=FRAGMENT_SIZE, cache=True, tiny_subset=SANITY_SAMPLES,
)

model = build_model('depthwisecnn', num_classes=train_ds.num_classes,
                    variant=VARIANT, fragment_size=FRAGMENT_SIZE)
print(f'Model: {model.name}  params={model.num_parameters():,}')

tcfg = TrainerConfig(
    epochs=SANITY_EPOCHS, lr=LR, patience=SANITY_EPOCHS,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    amp=torch.cuda.is_available(), seed=SEED,
)
trainer = Trainer(model, tcfg, run_dir=OUTPUTS_DIR / 'sanity')

loader_tr = build_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
loader_va = build_dataloader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

hist = trainer.fit(loader_tr, loader_va)
print(f'\nSanity done. val_acc={hist.val_acc[-1]:.4f}  ✓')

del model, trainer, train_ds, val_ds
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

---
## § 5 — Full training

In [ ]:
import sys
if '/kaggle/working/deepcarv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deepcarv')

import gc
from pathlib import Path

import torch

from benchmarks.DepthwiseCNN.scripts.train import main as train_main

CANONICAL_ROOT = Path('/kaggle/working/data/FFT-75')
OUTPUTS_DIR    = Path('/kaggle/working/outputs')

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

run_name = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
run_dir  = OUTPUTS_DIR / run_name

train_main([
    '--data_dir',      str(CANONICAL_ROOT),
    '--fragment_size', str(FRAGMENT_SIZE),
    '--variant',       VARIANT,
    '--epochs',        str(EPOCHS),
    '--batch_size',    str(BATCH_SIZE),
    '--lr',            str(LR),
    '--seed',          str(SEED),
    '--run_dir',       str(run_dir),
    '--no_timestamp',
])

print(f'\nTraining complete -> {run_dir}')

---
## § 6 — Evaluation (test split)

In [ ]:
import sys
if '/kaggle/working/deepcarv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deepcarv')

import gc
import shutil
from pathlib import Path

import torch

from benchmarks.DepthwiseCNN.scripts.evaluate import main as eval_main

CANONICAL_ROOT = Path('/kaggle/working/data/FFT-75')
OUTPUTS_DIR    = Path('/kaggle/working/outputs')
CKPT_DIR       = Path('/kaggle/working/checkpoints')

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

run_name        = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
run_dir         = OUTPUTS_DIR / run_name
eval_dir        = OUTPUTS_DIR / f'{run_name}_eval'
train_best_ckpt = run_dir / 'checkpoint_best.pt'
canonical_ckpt  = CKPT_DIR / f'best_{run_name}.pt'

if not train_best_ckpt.exists():
    raise FileNotFoundError(
        f'Best checkpoint not found at {train_best_ckpt}.\n'
        f'Run § 5 (full training) first.'
    )

shutil.copy2(train_best_ckpt, canonical_ckpt)
print(f'Best checkpoint -> {canonical_ckpt}')

eval_main([
    '--checkpoint',    str(canonical_ckpt),
    '--data_dir',      str(CANONICAL_ROOT),
    '--fragment_size', str(FRAGMENT_SIZE),
    '--variant',       VARIANT,
    '--out_dir',       str(eval_dir),
    '--split',         'test',
    '--batch_size',    '1024',
])

print(f'\nEvaluation complete -> {eval_dir}')

---
## § 7 — Results summary + bundle

In [ ]:
import json
from pathlib import Path

OUTPUTS_DIR = Path('/kaggle/working/outputs')

run_name = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
eval_dir = OUTPUTS_DIR / f'{run_name}_eval'

summary_path = eval_dir / 'summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    print('=== Evaluation Summary ===')
    for k, v in summary.items():
        print(f'  {k:<30} {v:.6f}' if isinstance(v, float) else f'  {k:<30} {v}')
else:
    print('summary.json not found:', eval_dir)

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUTS_DIR = Path('/kaggle/working/outputs')
run_name    = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
eval_dir    = OUTPUTS_DIR / f'{run_name}_eval'

per_class_path = eval_dir / 'per_class_metrics.csv'
if per_class_path.exists():
    pc = pd.read_csv(per_class_path, index_col='class')
    print('\n=== Top 10 classes by F1 ===')
    print(pc.sort_values('f1', ascending=False).head(10).to_string())
    print('\n=== Bottom 10 classes by F1 ===')
    print(pc.sort_values('f1', ascending=True).head(10).to_string())

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import Image, display
from pathlib import Path

OUTPUTS_DIR = Path('/kaggle/working/outputs')
run_name    = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
eval_dir    = OUTPUTS_DIR / f'{run_name}_eval'
run_dir     = OUTPUTS_DIR / run_name

cm_path = eval_dir / 'confusion_matrix.csv'
if cm_path.exists():
    cm = pd.read_csv(cm_path).values
    cm_norm = cm.astype(float)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_norm /= row_sums

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm_norm, ax=ax, cmap='Blues', vmin=0, vmax=1,
                xticklabels=False, yticklabels=False,
                cbar_kws={'label': 'Fraction correct'})
    ax.set_xlabel('Predicted class', fontsize=12)
    ax.set_ylabel('True class', fontsize=12)
    ax.set_title(
        f'DepthwiseCNN ({VARIANT}) — FFT-75 {FRAGMENT_SIZE}B\nNormalised confusion matrix',
        fontsize=13)
    fig.tight_layout()
    out_png = eval_dir / 'confusion_matrix_heatmap.png'
    plt.savefig(out_png, dpi=150)
    plt.show()
    print('Saved:', out_png)

for curve in ('loss_curve.png', 'accuracy_curve.png'):
    src = run_dir / curve
    if src.exists():
        print(f'\n{curve}:')
        display(Image(filename=str(src)))

In [ ]:
import sys
if '/kaggle/working/deepcarv' not in sys.path:
    sys.path.insert(0, '/kaggle/working/deepcarv')

import json
import shutil
from pathlib import Path

from src.models.registry import build_model

WORKING_DIR = Path('/kaggle/working')
OUTPUTS_DIR = WORKING_DIR / 'outputs'
CKPT_DIR    = WORKING_DIR / 'checkpoints'

run_name = f'depthwisecnn_{VARIANT}_{FRAGMENT_SIZE}b'
eval_dir = OUTPUTS_DIR / f'{run_name}_eval'
run_dir  = OUTPUTS_DIR / run_name
ckpt     = CKPT_DIR / f'best_{run_name}.pt'

m = build_model('depthwisecnn', num_classes=75, variant=VARIANT, fragment_size=FRAGMENT_SIZE)
s = {}
if (eval_dir / 'summary.json').exists():
    with open(eval_dir / 'summary.json') as f:
        s = json.load(f)

print('\n' + '='*55)
print('  DepthwiseCNN Benchmark Card')
print('='*55)
print(f'  Variant         : {VARIANT}')
print(f'  Fragment size   : {FRAGMENT_SIZE} bytes')
print(f'  Parameters      : {m.num_parameters():,}')
print(f'  Test accuracy   : {s.get("accuracy", "N/A")}')
print(f'  Macro F1        : {s.get("macro_f1", "N/A")}')
print(f'  Weighted F1     : {s.get("weighted_f1", "N/A")}')
print('='*55)

bundle_dir = WORKING_DIR / f'DepthwiseCNN_{VARIANT}_{FRAGMENT_SIZE}b_bundle'
bundle_dir.mkdir(parents=True, exist_ok=True)
prefix = f'{FRAGMENT_SIZE}b_{VARIANT}'

for fname in ('metrics.json', 'summary.json', 'predictions.csv',
              'confusion_matrix.csv', 'per_class_metrics.csv',
              'classification_report.txt', 'confusion_matrix_heatmap.png'):
    src = eval_dir / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / f'{prefix}_{fname}')

for fname in ('loss_curve.png', 'accuracy_curve.png', 'lr_curve.png'):
    src = run_dir / fname
    if src.exists():
        shutil.copy2(src, bundle_dir / f'{prefix}_{fname}')

if ckpt.exists():
    shutil.copy2(ckpt, bundle_dir / ckpt.name)

zip_path = WORKING_DIR / f'DepthwiseCNN_{VARIANT}_{FRAGMENT_SIZE}b_bundle.zip'
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)
print(f'\nBundle: {zip_path}  ({zip_path.stat().st_size/1e6:.1f} MB)')
for f in sorted(bundle_dir.iterdir()):
    print(f'  {f.name}')